# Aliado Libre — fine-tuning LoRA en GPU (Google Colab)

Este notebook entrena el mismo adaptador LoRA que `finetune/entrenar.py` corre localmente, pero en la GPU gratuita de Colab (mucho más rápido que el CPU de la laptop).

**Antes de correr:** `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → GPU (T4 alcanza).

**Pasos:**
1. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
2. Cuando la celda de carga pida el archivo, sube `finetune/data/entrenamiento.jsonl` desde tu PC.
3. Elige qué modelo entrenar en la celda de configuración (`0.5B` o `1.5B`) — puedes correr el notebook dos veces, una por cada uno.
4. Al final se descarga un `.zip` con el adaptador entrenado (`modelo_lora_05b.zip` o `modelo_lora_15b.zip`).
5. Descomprime ese zip dentro de `finetune/` en tu proyecto local, reemplazando la carpeta correspondiente.

In [ ]:
!pip install -q peft trl accelerate datasets transformers

## Configuración — elige el modelo a entrenar

In [ ]:
# Cambia esto según qué experimento quieras correr en esta sesión
MODELO_BASE = "Qwen/Qwen2.5-0.5B-Instruct"  # o "Qwen/Qwen2.5-1.5B-Instruct"
NOMBRE_SALIDA = "modelo_lora_05b"             # o "modelo_lora_15b"
NUM_EPOCHS = 3  # en GPU esto es rápido, se puede volver a las 3 épocas originales

## Subir el dataset
Sube `finetune/data/entrenamiento.jsonl` (50 ejemplos, ya preparado en el proyecto local).

In [ ]:
from google.colab import files
subido = files.upload()
RUTA_DATASET = list(subido.keys())[0]
print(f"Dataset cargado: {RUTA_DATASET}")

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "No hay GPU activa — revisa Entorno de ejecución > Cambiar tipo de entorno de ejecución"
print("GPU:", torch.cuda.get_device_name(0))

ejemplos = [json.loads(l) for l in open(RUTA_DATASET, encoding="utf-8") if l.strip()]
textos = [f"{e['prompt']}{e['completion']}" for e in ejemplos]
dataset = Dataset.from_dict({"text": textos})
print(f"{len(dataset)} ejemplos cargados")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, dtype="bfloat16").cuda()

config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
modelo = get_peft_model(modelo, config_lora)
modelo.print_trainable_parameters()

In [ ]:
argumentos = SFTConfig(
    output_dir="checkpoints",
    bf16=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to=[],
    max_length=1024,
    dataset_text_field="text",
)

entrenador = SFTTrainer(
    model=modelo,
    args=argumentos,
    train_dataset=dataset,
    processing_class=tokenizer,
)

entrenador.train()

In [ ]:
modelo.save_pretrained(NOMBRE_SALIDA)
tokenizer.save_pretrained(NOMBRE_SALIDA)
print(f"Adaptador guardado en {NOMBRE_SALIDA}/")

## Descargar el resultado
Comprime el adaptador y lo descarga — descomprímelo en `finetune/` dentro de tu proyecto local (reemplazando la carpeta `finetune/{NOMBRE_SALIDA}/`).

In [ ]:
import shutil
shutil.make_archive(NOMBRE_SALIDA, "zip", NOMBRE_SALIDA)
files.download(f"{NOMBRE_SALIDA}.zip")